# LAMPS Full Multi-Agent Pipeline — D2 Evaluation

| Agent | Backend | Vai trò |
|---|---|---|
| **Extractor** | **Ollama (deepseek-v4-flash:cloud)** | Semantic file selection |
| **Classifier** | **Fine-tuned CodeBERT** | Per-file malicious/benign |
| **Verdict** | **Ollama (deepseek-v4-flash:cloud)** | Conservative aggregation + rationale |

Yêu cầu trên Drive:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- `NT230/data/d2/files.jsonl` và `packages.jsonl`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, sys, json
from pathlib import Path
from collections import defaultdict

DRIVE_D1 = '/content/drive/My Drive/NT230/data/d1'
DRIVE_D2 = '/content/drive/My Drive/NT230/data/d2'

!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, '/content/NT230/src')

# model.bin
os.makedirs('/content/saved_models/checkpoint-best-acc', exist_ok=True)
shutil.copy(f'{DRIVE_D1}/saved_models/checkpoint-best-acc/model.bin',
            '/content/saved_models/checkpoint-best-acc/model.bin')
print('✅ model.bin:', round(os.path.getsize('/content/saved_models/checkpoint-best-acc/model.bin')/1e6), 'MB')

# D2 data
shutil.copy(f'{DRIVE_D2}/files.jsonl',    '/content/d2_files.jsonl')
shutil.copy(f'{DRIVE_D2}/packages.jsonl', '/content/d2_packages.jsonl')
print('✅ files:', sum(1 for _ in open('/content/d2_files.jsonl')),
      '| packages:', sum(1 for _ in open('/content/d2_packages.jsonl')))

In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm ollama

# Load OLLAMA_API_KEY từ Drive .env
import re
env_path = '/content/drive/My Drive/NT230/data/.env'
try:
    for line in open(env_path).read().splitlines():
        m = re.match(r'^\$env:(\w+)\s*=\s*["\']?([^"\']+)["\']?', line.strip())
        if m: os.environ.setdefault(m.group(1), m.group(2).strip())
    print('✅ .env loaded, OLLAMA_API_KEY:', 'SET' if os.getenv('OLLAMA_API_KEY') else 'NOT FOUND')
except FileNotFoundError:
    print('⚠️  .env not found — set OLLAMA_API_KEY manually if needed')

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU')

# Test Ollama connection
from ollama import chat
resp = chat(model='deepseek-v4-flash:cloud', messages=[{'role':'user','content':'Hello!'}])
print('✅ Ollama connected:', resp.message.content[:60])

In [ ]:
# ── Khởi tạo 3 agents ────────────────────────────────────────────────────────
from lamps.llms.ollama_client import OllamaClient
from lamps.agents.extractor import LLMExtractorAgent, ExtractedFile
from lamps.agents.classifier import ClassifierAgent
from lamps.agents.verdict import VerdictAgent
from lamps.evaluation.metrics import classification_report, format_report
from lamps.utils import read_jsonl

llm = OllamaClient(
    model='deepseek-v4-flash:cloud',
    host='https://api.ollama.com',
    headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY', '')}
)

extractor_agent  = LLMExtractorAgent(llm=llm)          # Ollama — semantic filter
classifier_agent = ClassifierAgent(
    checkpoint='/content/saved_models/checkpoint-best-acc/model.bin',
    batch_size=64
)                                                       # CodeBERT — classify
verdict_agent    = VerdictAgent(llm=llm)                # Ollama — rationale

print('✅ All 3 agents ready')

In [ ]:
# ── Load D2 ──────────────────────────────────────────────────────────────────
file_records    = list(read_jsonl(Path('/content/d2_files.jsonl')))
package_records = list(read_jsonl(Path('/content/d2_packages.jsonl')))

files_by_pkg = defaultdict(list)
for r in file_records:
    files_by_pkg[r['package']].append(
        ExtractedFile(package=r['package'], path=Path('<memory>'),
                      rel_path=str(r.get('path','')), source=str(r['func']))
    )
print(f'Packages: {len(package_records)} | Total files: {len(file_records)}')

In [ ]:
# ── Full pipeline: Extractor → Classifier → Verdict ──────────────────────────
from tqdm import tqdm

y_pkg_true, y_pkg_pred = [], []
pkg_results = []
total_before, total_after = 0, 0

for pkg in tqdm(package_records, desc='Packages'):
    name       = pkg['package']
    true_label = int(pkg['label'])
    all_files  = files_by_pkg.get(name, [])

    # Agent 1: Extractor — Ollama semantic filtering
    filtered = extractor_agent.filter(name, all_files)
    total_before += len(all_files)
    total_after  += len(filtered)

    # Agent 2: Classifier — CodeBERT per-file
    classifications = classifier_agent.classify_files(filtered)

    # Agent 3: Verdict — conservative aggregation + Ollama rationale
    verdict = verdict_agent.aggregate(name, classifications)

    y_pkg_true.append(true_label)
    y_pkg_pred.append(verdict.target)
    pkg_results.append({
        'package'          : name,
        'target'           : true_label,
        'predicted'        : verdict.target,
        'n_files_before'   : len(all_files),
        'n_files_after'    : len(filtered),
        'n_malicious_files': len(verdict.malicious_files),
        'rationale'        : verdict.rationale,
    })

print(f'Extractor: {total_before} → {total_after} files ({total_after/max(total_before,1)*100:.1f}% kept)')

In [ ]:
# ── Results ───────────────────────────────────────────────────────────────────
report = classification_report(y_pkg_true, y_pkg_pred)
print('=== Package-level — Full Multi-Agent Pipeline ===')
print(format_report(report))

In [ ]:
# ── Save về Drive ─────────────────────────────────────────────────────────────
os.makedirs('/content/results_full', exist_ok=True)
with open('/content/results_full/package_report.json','w') as f:
    json.dump(report.to_dict(), f, indent=2)
with open('/content/results_full/package_predictions.jsonl','w') as f:
    f.write('\n'.join(json.dumps(p, ensure_ascii=False) for p in pkg_results))

shutil.copytree('/content/results_full', f'{DRIVE_D2}/results_full_pipeline', dirs_exist_ok=True)
print('✅ Saved to Drive: NT230/data/d2/results_full_pipeline/')